# NLP: Asistente RAG sobre la Guía de Derechos ANSES

---

**Corpus elegido:** *"ANSES en cada etapa de tu vida"* — Guía de derechos de la Administración Nacional de la Seguridad Social (Argentina), edición agosto 2023. El documento tiene 116 páginas (~68.000 caracteres) y cubre prestaciones sociales organizadas por etapa de vida: embarazo, niñez, adultez y personas mayores.  

Este documento resultó interesante para realizar un RAG, ya que la normativa de seguridad social en Argentina es extensa y suele generar confusión.


## 0. Instalación de dependencias

In [ ]:
!pip install -q --upgrade \
    transformers accelerate \
    pdfplumber sentence-transformers faiss-cpu gdown


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 78.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import pdfplumber
import gdown
import statistics
import numpy as np
import faiss
import torch
from collections import Counter
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, pipeline
import transformers
transformers.logging.set_verbosity_error()

from google.colab import userdata


---
## Fase 1 — Corpus y Chunking

### 1.1 Carga y limpieza del corpus

Se descarga el PDF de la guía ANSES desde Google Drive y se extrae el texto página por página. Luego se agrupan las páginas en **documentos temáticos** (8) según las secciones del libro (identificadas por sus títulos de capítulo).

In [ ]:
# Descargar Corpus: PDF guía del ANSES
FILE_ID = "1LrvV8SxF45r-iWnWUAfpXyy1gwzozp43"
PDF_PATH = "anses_guia.pdf"
gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", PDF_PATH, quiet=False)
print("PDF descargado.")

Downloading...
From: https://drive.google.com/uc?id=1LrvV8SxF45r-iWnWUAfpXyy1gwzozp43
To: /content/anses_guia.pdf
100%|██████████| 11.5M/11.5M [00:00<00:00, 96.7MB/s]

PDF descargado.


**Filtrado de Contenido:** Se excluyen páginas irrelevantes (índices y contraportadas) y se descartan fragmentos menores a 30 caracteres para evitar procesar metadatos aislados.

**Deduplicación de Caracteres:** Se implementa la función para corregir errores de extracción. En este PDF, las negritas suelen interpretarse como caracteres duplicados por pdfplumber (ej: "uunn" → "un"). Se usan expresiones regulares para colapsarlos, protegiendo las letras dobles válidas del español (ll, rr, cc).

**Normalización Estructural:** Se eliminan números de página aislados.

- Se colapsan espacios múltiples y se estandarizan saltos de línea para mantener la cohesión de los párrafos.

In [ ]:
# Deduplicaión
def reparar_ocr_duplicado(texto: str) -> str:
    """
    Corrige palabras donde el OCR duplicó cada letra.
    Conserva duplicaciones válidas del español:
    ll, rr, cc
    """

    palabras = texto.split()
    nuevas_palabras = []

    for palabra in palabras:
        # palabra corta no tocar
        if len(palabra) < 4:
            nuevas_palabras.append(palabra)
            continue
        reconstruida = ""
        i = 0
        while i < len(palabra):
            # Detecta letras duplicadas consecutivas
            if (
                i + 1 < len(palabra)
                and palabra[i].lower() == palabra[i + 1].lower()
            ):
                letra = palabra[i]

                # Mantener duplicaciones válidas comunes
                if letra.lower() in ["l", "r", "c"]:
                    reconstruida += letra * 2
                else:
                    reconstruida += letra
                i += 2
            else:
                reconstruida += palabra[i]
                i += 1
        nuevas_palabras.append(reconstruida)
    return " ".join(nuevas_palabras)


# Limpieza general
def limpiar_texto(texto: str) -> str:
    """
    Limpieza conservadora orientada a PDFs/OCR.
    """
    if not texto:
        return ""
    # eliminar números de página aislados
    texto = re.sub(r'(?m)^\s*\d+\s*$', '', texto)
    # unir palabras cortadas por salto de línea
    texto = re.sub(r'(\w)-\n(\w)', r'\1\2', texto)
    # funcion para deduplicar
    texto = reparar_ocr_duplicado(texto)
    # corregir espacios excesivos
    texto = re.sub(r'[ \t]+', ' ', texto)
    # Mantener párrafos legibles
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    # Limpiar espacios alrededor de saltos
    texto = re.sub(r' *\n *', '\n', texto)
    #limpiar bullets rotos
    texto = re.sub(r'•\s+', '• ', texto)

    return texto.strip()


# EXTRACCIÓN DEL CONTENIDO DEL PDF
PAGINAS_EXCLUIR = set(range(6, 10)) | set(range(112, 116))

paginas = []

with pdfplumber.open(PDF_PATH) as pdf:

    for i, page in enumerate(pdf.pages):

        num_pag = i + 1

        # Saltar páginas innecesarias
        if num_pag in PAGINAS_EXCLUIR:
            continue

        texto = page.extract_text()

        # Ignorar páginas vacías
        if not texto or len(texto.strip()) < 30:
            continue

        # Limpiar texto
        texto_limpio = limpiar_texto(texto)

        paginas.append({
            "pagina": num_pag,
            "texto": texto_limpio
        })



# Verificación
print(f"Páginas con contenido extraído: {len(paginas)}")
if paginas:
  print(f"\nEjemplo — página {paginas[1]['pagina']}:")
  print(paginas[15]['texto'][:400])

Páginas con contenido extraído: 93

Ejemplo — página 4:
mi ANSES ¿Qué es? Canal de atención digital que permite realizar, de forma autogestionada (durante las 24 horas del día), trámites y consultas, sin necesidad de ir a una oficina. Para ingresar desde la web del organismo o descargando la app en el celular, solo se necesita tener CUIL y Clave de la Seguridad Social. ¿Qué consultas y trámites se pueden hacer a través de mi ANSES? CONSULTAS • Recibos 


In [ ]:
# Agrupar páginas en documentos temáticos
SECCIONES_LIMPIAS = [
    ("Presentación y CUIL",                  1,   21),
    ("Actualización de Datos y Clave",       22,   33),
    ("Embarazo, Nacimiento y Maternidad",    34,   43),
    ("Niñez, Adolescencia y Juventud",       44,   61),
    ("Adultez y Trabajo",                    63,   77),
    ("Personas Mayores — Jubilación",        79,   95),
    ("Pensiones y Prestaciones Especiales",  98,  105),
    ("ANSES en números",                    107,  111),
]

# Crear índice página → texto (ya filtrado por la celda anterior)
pag2txt = {p['pagina']: p['texto'] for p in paginas}

documentos = []
for titulo, inicio, fin in SECCIONES_LIMPIAS:
    texto_seccion = "\n\n".join(
        pag2txt[pg] for pg in range(inicio, fin + 1) if pg in pag2txt
    )
    if texto_seccion:
        documentos.append({"titulo": titulo, "texto": texto_seccion})

suma = 0
print(f"Documentos construidos: {len(documentos)}")
for doc in documentos:
    print(f"  • {doc['titulo']}: {len(doc['texto']):,} caracteres")
    suma += len(doc['texto'])
print(f"\n{suma} caracteres totales" )

Documentos construidos: 8
  • Presentación y CUIL: 7,592 caracteres
  • Actualización de Datos y Clave: 5,697 caracteres
  • Embarazo, Nacimiento y Maternidad: 6,513 caracteres
  • Niñez, Adolescencia y Juventud: 10,367 caracteres
  • Adultez y Trabajo: 9,099 caracteres
  • Personas Mayores — Jubilación: 10,655 caracteres
  • Pensiones y Prestaciones Especiales: 5,238 caracteres
  • ANSES en números: 755 caracteres

55916 caracteres totales


In [ ]:
longitudes = [len(d['texto']) for d in documentos]
total_chars = sum(longitudes)

print(f"{'─'*45}")
print(f"Estadísticas del corpus")
print(f"{'─'*45}")
print(f"  Documentos          : {len(documentos)}")
print(f"  Total de caracteres : {total_chars:,}")
print(f"  Promedio por doc    : {statistics.mean(longitudes):,.0f}")
print(f"  Mínimo              : {min(longitudes):,}")
print(f"  Máximo              : {max(longitudes):,}")
print(f"{'─'*45}")

─────────────────────────────────────────────
Estadísticas del corpus
─────────────────────────────────────────────
  Documentos          : 8
  Total de caracteres : 55,916
  Promedio por doc    : 6,990
  Mínimo              : 755
  Máximo              : 10,655
─────────────────────────────────────────────


### 1.2 Estrategias de Chunking

Para fragmentar el corpus, se evalúan dos enfoques con el fin de encontrar el equilibrio entre granularidad y preservación del contexto:

- **Estrategia A (Fixed-size)**: Una división rígida cada 500 caracteres con un overlap de 100. Es sencilla, pero corre el riesgo de fragmentar oraciones clave o listas de requisitos a la mitad. Si bien el overlap mitiga esto, no lo elimina.

- **Estrategia B (Paragraph-aware)**: Esta técnica prioriza la estructura semántica del documento. Para este caso, se eligió el modelo BAAI/bge-m3 para tokenizar y agrupar párrafos completos (separados por doble salto de línea). Solo fragmenta si se supera el límite de 300 tokens, manteniendo un solapamiento (overlap) de 120 tokens para no perder la conexión entre bloques. Esta combinación es la que mejor funcionó.

En principio, el chunking paragraph-aware debería rendir mejor en la etapa de retrieval para este corpus, ya que permite que los embeddings representen conceptos completos y no fragmentos arbitrarios de texto.

In [ ]:
# Estrategia A: Fixed-size con overlap

def chunk_fixed(documentos, tam=500, overlap=100):
    """Divide cada documento en ventanas de `tam` caracteres con `overlap` de solapamiento."""
    chunks = []
    for doc in documentos:
        texto = doc['texto']
        inicio = 0
        while inicio < len(texto):
            fin = min(inicio + tam, len(texto))
            chunks.append({
                "texto":    texto[inicio:fin],
                "doc":      doc['titulo'],
                "estrategia": "fixed"
            })
            if fin == len(texto):
                break
            inicio += tam - overlap
    return chunks

chunks_A = chunk_fixed(documentos)
print(f"Chunks (Fixed-size): {len(chunks_A)}")
print(f"Longitud media: {sum(len(c['texto']) for c in chunks_A) / len(chunks_A):.0f} chars")

Chunks (Fixed-size): 141
Longitud media: 491 chars


In [ ]:
# Estrategia B: Paragraph-aware con overlap

MODEL_NAME = "BAAI/bge-m3"
chunk_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_paragraph(documentos, max_tokens=300, overlap_tokens=120):
    chunks = []
    for doc in documentos:
        parrafos = [p.strip() for p in doc['texto'].split("\n\n") if p.strip()]
        current_ids = []
        for para in parrafos:
            para_ids = chunk_tokenizer.encode(para, add_special_tokens=False)
            if len(current_ids) + len(para_ids) <= max_tokens:
                current_ids.extend(para_ids)
            else:
                if current_ids:
                    chunks.append({
                        "texto": chunk_tokenizer.decode(current_ids, skip_special_tokens=True),
                        "doc": doc['titulo'],
                        "estrategia": "paragraph"
                    })
                current_ids = current_ids[-overlap_tokens:] + para_ids
        if current_ids:
            chunks.append({
                "texto": chunk_tokenizer.decode(current_ids, skip_special_tokens=True),
                "doc": doc['titulo'],
                "estrategia": "paragraph"
            })
    return chunks
chunks_B = chunk_paragraph(documentos)
print(f"Chunks (Paragraph-aware): {len(chunks_B)}")
print(f"Longitud media: {sum(len(c['texto']) for c in chunks_B) / len(chunks_B):.0f} chars")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Chunks (Paragraph-aware): 78
Longitud media: 1181 chars


In [ ]:
# Comparación visual de chunks

print("=" * 60)
print("ESTRATEGIA A — Fixed-size (primeros 3 chunks)")
print("=" * 60)
for c in chunks_A[:3]:
    print(f"[Doc: {c['doc']}]")
    print(c['texto'][:500])
    print("-" * 40)

print()
print("=" * 60)
print("ESTRATEGIA B — Paragraph-aware (primeros 3 chunks)")
print("=" * 60)
for c in chunks_B[:3]:
    print(f"[Doc: {c['doc']}]")
    print(c['texto'][:1000])
    print("-" * 40)

ESTRATEGIA A — Fixed-size (primeros 3 chunks)
[Doc: Presentación y CUIL]
Guía de derechos En cada etapa de tu vida

Autoridades Presidente de la Nación Dr. Alberto Fernández Vicepresidenta de la Nación Dra. Cristina Fernández de Kirchner Directora Ejecutiva de la Administración Nacional de la Seguridad Social Lic. Fernanda Raverta

ANSES en cada etapa de tu vida ANSES es el corazón del Estado Nacional. Es el organismo que está presente en cada etapa de la vida, acompañando a las argentinas y argentinos con políticas de Seguridad Social desde antes de nacer, durante l
----------------------------------------
[Doc: Presentación y CUIL]
ando a las argentinas y argentinos con políticas de Seguridad Social desde antes de nacer, durante la infancia, adolescencia y adultez. Esta guía de derechos está organizada de acuerdo a cada una de estas etapas de la vida y contiene toda la información necesaria, explicada de manera sencilla, sobre las prestaciones con las que ANSES acompaña a las persona

---
## Fase 2 — Embeddings e Indexación FAISS

`BAAI/bge-m3` fue el embedder multilingüe que mejor rindió en las pruebas realizadas. También se evaluaron `paraphrase-multilingual-mpnet-base-v2`, `intfloat/multilingual-e5-base` y `hiiamsid/sentence_similarity_spanish_es`, que tuvo buenos resultados en español, aunque bge-m3 mostró un rendimiento ligeramente superior en general.

In [ ]:
MODEL_NAME = "BAAI/bge-m3"
embedder = SentenceTransformer(MODEL_NAME)
print(f"Modelo cargado: {MODEL_NAME}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Modelo cargado: BAAI/bge-m3


In [ ]:
def construir_indice(chunks):
    """Genera embeddings normalizados y construye un índice FAISS con similitud coseno."""
    textos = [c['texto'] for c in chunks]
    embs = embedder.encode(textos, normalize_embeddings=True,
                           batch_size=64, show_progress_bar=True)
    embs = embs.astype(np.float32)
    dim = embs.shape[1]
    indice = faiss.IndexFlatIP(dim)
    indice.add(embs)
    return indice

print("Construyendo índice A (fixed-size)...")
indice_A = construir_indice(chunks_A)
print(f"  → {indice_A.ntotal} vectores indexados")

print("Construyendo índice B (paragraph-aware)...")
indice_B = construir_indice(chunks_B)
print(f"  → {indice_B.ntotal} vectores indexados")


Construyendo índice A (fixed-size)...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

  → 141 vectores indexados
Construyendo índice B (paragraph-aware)...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  → 78 vectores indexados


In [ ]:
def buscar(query, indice, chunks, k=5):
    """Devuelve los top-k chunks más relevantes para la query con su score coseno."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = indice.search(q_emb, k)
    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        resultados.append({
            "score": float(score),
            "doc":   chunks[idx]['doc'],
            "texto": chunks[idx]['texto']
        })
    return resultados

# Prueba rápida de retrieval
query_prueba = "¿Hasta qué edad cobran la Asignación Universal por Hijo los hijos?"
print(f"Query: {query_prueba}\n")
res = buscar(query_prueba, indice_B, chunks_B, k=3)
for r in res:
    print(f"[Score: {r['score']:.3f}] [{r['doc']}]")
    print(r['texto'][:700])
    print()


Query: ¿Hasta qué edad cobran la Asignación Universal por Hijo los hijos?

[Score: 0.729] [Niñez, Adolescencia y Juventud]
aciones familiares o universales por cada hija o hijo en edad escolar (a partir de los 45 días y hasta los 17 años inclusive) o sin límite de edad por hija o hijo con discapacidad. Se certifica en la escuela y se presenta en las oficinas de ANSES. Se abona y se certifica una vez al año CCon lla nueva LLey de Movillidad, todas llas prrestacciones de ANSES tienen un aumento trrimestrrall que se rrealliza en marrzo, junio, septiembrre yy dicciembrre de ccada año. Asignación Universal por Hija e Hijo (AUH) Pago mensual que tiene como objetivo igualar las oportunidades de todas las niñas, niños y adolescentes del país, garantizando su derecho a la educación, a la salud y a la Seguridad Social. ¿A

[Score: 0.672] [Niñez, Adolescencia y Juventud]
Beneficios ANSES Niñez, Adolescencia y Juventud • Ser argentina o argentino, residir en el país y tener DNI • Si es extranjera 

---
## Fase 3 — Evaluación del Retrieval (hit@k)

In [ ]:
# Set de evaluación: 10 preguntas que combinan los dos conjuntos originales
# Cubren todas las secciones del corpus para un retrieval no trivial :)

preguntas_eval = [
    {
        "pregunta": "¿Qué es la Asignación Universal por Hijo y a quiénes está destinada?",
        "doc_esperado": "Niñez, Adolescencia y Juventud"
    },
    {
        "pregunta": "¿Cuáles son los requisitos para tramitar la jubilación en ANSES?",
        "doc_esperado": "Personas Mayores — Jubilación"
    },
    {
        "pregunta": "¿A partir de qué semana de embarazo se puede cobrar la Asignación por Embarazo para Protección Social?",
        "doc_esperado": "Embarazo, Nacimiento y Maternidad"
    },
    {
        "pregunta": "¿Cómo se obtiene la Clave de la Seguridad Social?",
        "doc_esperado": "Actualización de Datos y Clave"
    },
    {
        "pregunta": "¿Qué es el CUIL y para qué se necesita?",
        "doc_esperado": "Presentación y CUIL"
    },

    {
        "pregunta": "¿Qué documentos se necesitan para cobrar la Asignación Familiar por Maternidad?",
        "doc_esperado": "Embarazo, Nacimiento y Maternidad"
    },
    {
        "pregunta": "¿Hasta qué edad cobran la Asignación Universal por Hijo los hijos?",
        "doc_esperado": "Niñez, Adolescencia y Juventud"
    },
    {
        "pregunta": "¿Cuáles son los requisitos para cobrar la Prestación por Desempleo?",
        "doc_esperado": "Adultez y Trabajo"
    },
    {
        "pregunta": "¿Qué es la Pensión Universal para la Adulta Mayor?",
        "doc_esperado": "Personas Mayores — Jubilación"
    },
    {
        "pregunta": "¿Qué trámites se pueden hacer desde la app mi ANSES?",
        "doc_esperado": "Actualización de Datos y Clave"
    },
]

print(f"Set de evaluación: {len(preguntas_eval)} preguntas")
for i, p in enumerate(preguntas_eval, 1):
    print(f"  P{i:02d}: {p['pregunta'][:65]}... → [{p['doc_esperado']}]")


Set de evaluación: 10 preguntas
  P01: ¿Qué es la Asignación Universal por Hijo y a quiénes está destina... → [Niñez, Adolescencia y Juventud]
  P02: ¿Cuáles son los requisitos para tramitar la jubilación en ANSES?... → [Personas Mayores — Jubilación]
  P03: ¿A partir de qué semana de embarazo se puede cobrar la Asignación... → [Embarazo, Nacimiento y Maternidad]
  P04: ¿Cómo se obtiene la Clave de la Seguridad Social?... → [Actualización de Datos y Clave]
  P05: ¿Qué es el CUIL y para qué se necesita?... → [Presentación y CUIL]
  P06: ¿Qué documentos se necesitan para cobrar la Asignación Familiar p... → [Embarazo, Nacimiento y Maternidad]
  P07: ¿Hasta qué edad cobran la Asignación Universal por Hijo los hijos... → [Niñez, Adolescencia y Juventud]
  P08: ¿Cuáles son los requisitos para cobrar la Prestación por Desemple... → [Adultez y Trabajo]
  P09: ¿Qué es la Pensión Universal para la Adulta Mayor?... → [Personas Mayores — Jubilación]
  P10: ¿Qué trámites se pueden hacer desde la a

In [ ]:
def evaluar_retrieval(preguntas, indice, chunks, ks=(1, 3, 5)):
    """Calcula hit@k para un set de preguntas anotadas."""
    hits = {k: 0 for k in ks}
    for item in preguntas:
        resultados = buscar(item['pregunta'], indice, chunks, k=max(ks))
        docs_recuperados = [r['doc'] for r in resultados]
        for k in ks:
            if item['doc_esperado'] in docs_recuperados[:k]:
                hits[k] += 1
    n = len(preguntas)
    return {k: hits[k] / n for k in ks}

print("Evaluando índice A (fixed-size)...")
metricas_A = evaluar_retrieval(preguntas_eval, indice_A, chunks_A)

print("Evaluando índice B (paragraph-aware)...")
metricas_B = evaluar_retrieval(preguntas_eval, indice_B, chunks_B)

print()
print(f"{'Estrategia':<25} {'hit@1':>8} {'hit@3':>8} {'hit@5':>8}")
print("-" * 52)
print(f"{'Fixed-size (A)':<25} {metricas_A[1]:>8.2f} {metricas_A[3]:>8.2f} {metricas_A[5]:>8.2f}")
print(f"{'Paragraph-aware (B)':<25} {metricas_B[1]:>8.2f} {metricas_B[3]:>8.2f} {metricas_B[5]:>8.2f}")

Evaluando índice A (fixed-size)...
Evaluando índice B (paragraph-aware)...

Estrategia                   hit@1    hit@3    hit@5
----------------------------------------------------
Fixed-size (A)                0.90     1.00     1.00
Paragraph-aware (B)           1.00     1.00     1.00


A partir de la comparativa visual, con un Hit@1 levemente mayor, se eligió la Estrategia B por tres razones clave:

1. **Integridad Semántica:** A diferencia de Fixed-size, que corta palabras y oraciones al azar, este método mantiene los requisitos de ANSES agrupados, evitando fragmentos incompletos.

2. **Calidad de Embeddings:** El modelo BGE-M3 genera mejores vectores cuando el texto tiene sentido completo. Un párrafo íntegro es más fácil de indexar y recuperar que un recorte arbitrario.

3. **Prevención de Alucinaciones:** Al entregarle al LLM párrafos coherentes y no frases cortadas, se minimiza el riesgo de que el modelo invente información para completar la respuesta.

Conclusión: Para este corpus, el respeto a la estructura del párrafo garantiza que el asistente recupere respuestas, no solo fragmentos, lo que hace que funcione mejor.


---
## Fase 4 — Generación con LLM (RAG vs sin RAG)

In [ ]:
"""
Usamos el modelo gemma 2, de 2B de parámetros porque tuvo el mejor rendimiento,
frente a LiquidAI/LFM2.5-1.2B-Instruct o Qwen/Qwen2.5-1.5B-Instruct
"""

from transformers import pipeline
import torch

LLM_NAME = "google/gemma-2-2b-it"

# Gemma 2 es un modelo con acceso restringido: hay que aceptar la licencia en
# https://huggingface.co/google/gemma-2-2b-it y usar un token de Hugging Face.
# En Colab, guardalo en "Secretos" (ícono de llave, panel izquierdo) como HF_TOKEN.
try:
    token_hf = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass
    token_hf = getpass("Token de Hugging Face: ")

llm_pipeline = pipeline(
    task="text-generation",
    model=LLM_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=token_hf
)

# Evitar conflictos de max_length vs max_new_tokens
llm_pipeline.model.generation_config.max_length = None

print(f"LLM cargado: {LLM_NAME} en {llm_pipeline.model.device}")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

LLM cargado: google/gemma-2-2b-it en cuda:0


In [ ]:
def responder_con_rag(pregunta, k=5):
    """Recupera top-k chunks y genera una respuesta usando el corpus."""
    resultados = buscar(pregunta, indice_B, chunks_B, k=k)

    contexto = "\n\n".join(
        f"[Fragmento {i+1} — {r['doc']}]\n{r['texto']}"
        for i, r in enumerate(resultados)
    )

    # Instrucciones del sistema integradas
    instrucciones = (
        "Eres un asistente experto en prestaciones sociales de ANSES Argentina. "
        "Responde en español usando ÚNICAMENTE la información del CONTEXTO proporcionado. "

        "Reglas estrictas:\n"
        "- Corrige errores de ortografía. "
        "- Si el dato exacto (edad, monto, semanas, años de aportes, documentos) "
        "no aparece en el contexto, responde exactamente: "
        "'No tengo información suficiente en el contexto proporcionado.'\n"
        "- NO inventes cifras, fechas, documentos ni requisitos ausentes del contexto.\n"
        "- Respondé primero con el dato concreto, luego agregá contexto si corresponde.\n"
        "- Sé completo en las respuestas pero conciso, no te extiendas demás. \n"
        "- Evita repetir palabras consecutivas o estructuras redundantes.\n"
        "- Si una frase del contexto está mal redactada, reescríbela de forma natural.\n"
        "- En caso de que sea necesario escribir la respuesta en items, hazlo. \n"
    )

    # promt
    prompt_final = f"{instrucciones}\n\nCONTEXTO:\n{contexto}\n\nPREGUNTA: {pregunta}"

    messages = [
        {"role": "user", "content": prompt_final},
    ]

    response = llm_pipeline(
        messages,
        max_new_tokens=400,
        do_sample=False,
        temperature=None
    )

    # Retorna el contenido del último mensaje generado por el modelo
    return response[0]["generated_text"][-1]["content"]


def responder_sin_rag(pregunta):
    """Genera respuesta sin retrieval (solo conocimiento paramétrico)."""

    instrucciones = "Eres un asistente útil que responde en español de forma completa y precisa. No repitas frases."

    messages = [
        {"role": "user", "content": f"{instrucciones}\n\nPregunta: {pregunta}"},
    ]

    response = llm_pipeline(
        messages,
        max_new_tokens=400,
        do_sample=False,
    )
    return response[0]["generated_text"][-1]["content"]

print("Funciones de generación listas.")

Funciones de generación listas.


In [ ]:
# Prueba rápida con la primera pregunta para verificar funcionamiento
pregunta_test = preguntas_eval[0]['pregunta']
print(f'Probando: {pregunta_test}\n')
resp_test = responder_con_rag(pregunta_test)
print('RESPUESTA CON RAG:')
print(resp_test)


Probando: ¿Qué es la Asignación Universal por Hijo y a quiénes está destinada?

RESPUESTA CON RAG:
La Asignación Universal por Hijo (AUH) es un pago mensual que tiene como objetivo igualar las oportunidades de todas las niñas, niños y adolescentes del país, garantizando su derecho a la educación, a la salud y a la Seguridad Social. 

Está destinada a:

* Hijas e hijos de personas con trabajo registrado (incluye eventuales y de temporada)
* Hijas e hijos de personas que cobran por una ART
* Hijas e hijos de monotributistas
* Hijos e hijas de personas que cobran la Prestación por Desempleo
* Hijas e hijos de personas que cobran la Pensión Honorífica de Veteranas y Veteranos de Guerra
* Hijas e hijos de personas que cobran una PNC por Invalidez o asignación por trasplante
* Hijas e hijos de personas jubiladas y pensionadas 



In [ ]:
# Ejecutar las 10 preguntas con ambos modos

resultados_gen = []
for i, item in enumerate(preguntas_eval):
    print(f"[{i+1}/10] {item['pregunta'][:65]}...")
    r_rag    = responder_con_rag(item['pregunta'])
    r_sinrag = responder_sin_rag(item['pregunta'])
    resultados_gen.append({
        "pregunta":    item["pregunta"],
        "doc_esperado": item["doc_esperado"],
        "resp_rag":    r_rag,
        "resp_sinrag": r_sinrag,
        "anotacion_rag":    "",   # ← completar: C / P / I / A
        "anotacion_sinrag": ""    # ← completar: C / P / I / A
    })

print("\n✓ Generación completada.")


[1/10] ¿Qué es la Asignación Universal por Hijo y a quiénes está destina...
[2/10] ¿Cuáles son los requisitos para tramitar la jubilación en ANSES?...
[3/10] ¿A partir de qué semana de embarazo se puede cobrar la Asignación...
[4/10] ¿Cómo se obtiene la Clave de la Seguridad Social?...
[5/10] ¿Qué es el CUIL y para qué se necesita?...
[6/10] ¿Qué documentos se necesitan para cobrar la Asignación Familiar p...
[7/10] ¿Hasta qué edad cobran la Asignación Universal por Hijo los hijos...
[8/10] ¿Cuáles son los requisitos para cobrar la Prestación por Desemple...
[9/10] ¿Qué es la Pensión Universal para la Adulta Mayor?...
[10/10] ¿Qué trámites se pueden hacer desde la app mi ANSES?...

✓ Generación completada.


In [ ]:
# Mostrar respuestas lado a lado
for i, r in enumerate(resultados_gen):
    display(Markdown(f"## Pregunta {i+1}"))
    display(Markdown(f"**P:** {r['pregunta']}"))
    display(Markdown(f"**Doc esperado:** `{r['doc_esperado']}`"))
    display(Markdown("### 🟢 CON RAG"))
    display(Markdown(r['resp_rag']))
    display(Markdown("### 🔴 SIN RAG"))
    display(Markdown(r['resp_sinrag']))
    print("\n" + "="*100 + "\n")


## Pregunta 1

**P:** ¿Qué es la Asignación Universal por Hijo y a quiénes está destinada?

**Doc esperado:** `Niñez, Adolescencia y Juventud`

### 🟢 CON RAG

La Asignación Universal por Hijo (AUH) es un pago mensual que tiene como objetivo igualar las oportunidades de todas las niñas, niños y adolescentes del país, garantizando su derecho a la educación, a la salud y a la Seguridad Social. 

Está destinada a:

* Hijas e hijos de personas con trabajo registrado (incluye eventuales y de temporada)
* Hijas e hijos de personas que cobran por una ART
* Hijas e hijos de monotributistas
* Hijos e hijas de personas que cobran la Prestación por Desempleo
* Hijas e hijos de personas que cobran la Pensión Honorífica de Veteranas y Veteranos de Guerra
* Hijas e hijos de personas que cobran una PNC por Invalidez o asignación por trasplante
* Hijas e hijos de personas jubiladas y pensionadas 


### 🔴 SIN RAG

... (60 outputs left out)

### 🟢 CON RAG

El trámite que se puede hacer desde la app mi ANSES son:

* Cambiar datos de contacto y domicilio
* Cambiar el lugar de cobro de prestaciones
* Certificado Escolar
* Ver credencial de jubilado
* Historia Laboral
* Fecha y lugar de cobro
* Certificación Negativa
* Efectuar reclamos, denuncias o sugerencias
* Tarifa Social para gas y transporte
* Constancia de CUIL 


### 🔴 SIN RAG

La app Mi ANSES te permite realizar diversos trámites, entre ellos:

* **Consultar tu situación laboral:**  Verificar tu situación laboral, incluyendo tu número de afiliado y tu historial de cotizaciones.
* **Solicitar y gestionar tu subsidio:**  Solicitar y gestionar tu subsidio de jubilación, pensión, maternidad, etc.
* **Solicitar y gestionar tu pago:**  Solicitar y gestionar tu pago de pensiones, jubilaciones, etc.
* **Gestionar tu cuenta:**  Modificar tu información personal, actualizar tu dirección, etc.
* **Consultar tu historial de pagos:**  Verificar el historial de pagos de tu cuenta.
* **Reportar cambios de datos:**  Informar sobre cambios en tu situación personal, como cambio de domicilio o de datos de contacto.
* **Solicitar información:**  Obtener información sobre los servicios que ofrece ANSES.
* **Contactar con ANSES:**  Enviar un mensaje a ANSES para resolver dudas o solicitar información.


Es importante recordar que la app Mi ANSES es una herramienta que te permite acceder a información y realizar trámites de forma rápida y sencilla. 


In [ ]:
# Pregunta random, a ver como responde el modelo

pregunta_random = "¿Qué es una banana?"
print(f'Probando: {pregunta_random}\n')
resp_random_rag = responder_con_rag(pregunta_random)
resp_random_srag = responder_sin_rag(pregunta_random)

print('RESPUESTA CON RAG:')
print(resp_random_rag)
print()
print('RESPUESTA SIN RAG:')
print(resp_random_srag)

Probando: ¿Qué es una banana?

RESPUESTA CON RAG:
No tengo información suficiente en el contexto proporcionado. 


RESPUESTA SIN RAG:
Una banana es una fruta tropical, originaria de Asia. Es una especie de planta llamada Musa acuminata, que produce una fruta alargada, con un color verde cuando está madura y un sabor dulce. 



### Anotación manual de cada respuesta

- **C** — Correcta
- **P** — Parcialmente correcta
- **I** — Incorrecta
- **A** — Alucinación

In [ ]:
#Anotaciones

anotaciones = [
    ("C", "A"),  # P1: ¿Qué es la AUH y a quiénes está destinada?
    ("C", "A"),  # P2: ¿Cuáles son los requisitos para la jubilación?
    ("C", "C"),  # P3: ¿A partir de qué semana se cobra la Asig. por Embarazo?
    ("C", "A"),  # P4: ¿Cómo se obtiene la Clave de la Seguridad Social?
    ("C", "C"),  # P5: ¿Qué es el CUIL y para qué se necesita?
    ("P", "A"),  # P6: ¿Qué documentos para la Asignación Familiar por Maternidad?
    ("P", "A"),  # P7: ¿Hasta qué edad cobran la AUH?
    ("C", "A"),  # P8: ¿Requisitos para la Prestación por Desempleo?
    ("C", "A"),  # P9: ¿Qué es la Pensión Universal para la Adulta Mayor?
    ("C", "A"),  # P10: ¿Qué trámites desde la app mi ANSES?
]

rag_anot    = [a[0] for a in anotaciones if a[0]]
sinrag_anot = [a[1] for a in anotaciones if a[1]]

def tabla_cpia(etiquetas, modo):
    cnt = Counter(etiquetas)
    n = len(etiquetas) or 1
    print(f"\n{modo}")
    print(f"  C (Correcta):          {cnt.get('C',0)}/{n} = {cnt.get('C',0)/n:.0%}")
    print(f"  P (Parcialmente):      {cnt.get('P',0)}/{n} = {cnt.get('P',0)/n:.0%}")
    print(f"  I (Incorrecta):        {cnt.get('I',0)}/{n} = {cnt.get('I',0)/n:.0%}")
    print(f"  A (Alucinación):       {cnt.get('A',0)}/{n} = {cnt.get('A',0)/n:.0%}")

tabla_cpia(rag_anot,    "CON RAG")
tabla_cpia(sinrag_anot, "SIN RAG")



CON RAG
  C (Correcta):          8/10 = 80%
  P (Parcialmente):      2/10 = 20%
  I (Incorrecta):        0/10 = 0%
  A (Alucinación):       0/10 = 0%

SIN RAG
  C (Correcta):          2/10 = 20%
  P (Parcialmente):      0/10 = 0%
  I (Incorrecta):        0/10 = 0%
  A (Alucinación):       8/10 = 80%


---
## Reporte Final

### 1. Corpus y objetivos

El corpus elegido es la *Guía de Derechos ANSES — En cada etapa de tu vida* (agosto 2023), un documento oficial de la Administración Nacional de la Seguridad Social argentina. El PDF tiene 116 páginas y aproximadamente 68.000 caracteres útiles tras la limpieza. Se dividió en 8 documentos temáticos siguiendo las secciones del libro: Presentación y CUIL, Actualización de Datos y Clave, Embarazo/Nacimiento/Maternidad, Niñez/Adolescencia/Juventud, Adultez y Trabajo, Personas Mayores — Jubilación, Pensiones y Prestaciones Especiales, y ANSES en Números. El objetivo del sistema es responder preguntas concretas sobre prestaciones de la seguridad social argentina: requisitos de acceso, montos, documentación necesaria, plazos y procedimientos de trámite.

---

### 2. Resultados del retrieval

Se evaluaron las dos estrategias de chunking sobre el mismo set de 10 preguntas anotadas, usando el embedder `BAAI/bge-m3` y similitud coseno con FAISS (`IndexFlatIP`).

| Estrategia | Chunks | Long. media | hit@1 | hit@3 | hit@5 |
|---|---|---|---|---|---|
| Fixed-size (A) | 141 | 491 chars | **0.90** | **1.00** | **1.00** |
| Paragraph-aware (B) | 78 | 1181 chars | **1.00** | **1.00** | **1.00** |

La estrategia Paragraph-aware obtuvo un hit@1 de 1.00 frente a 0.90 de Fixed-size, lo que significa que para *todas* las preguntas recuperó el documento correcto en primera posición. Fixed-size falló en al menos una pregunta a k=1 porque sus chunks de ~500 caracteres pueden cortar justo en medio de una prestación, haciendo que el embedding represente texto mezclado de dos secciones distintas y baje la similitud con la query. Paragraph-aware agrupa cada prestación completa en un único chunk, generando embeddings más representativos del concepto consultado.

A k=3 y k=5 ambas estrategias alcanzan 1.00 — el corpus tiene 8 documentos y cualquier documento relevante aparece entre los primeros 3 resultados. La diferencia en hit@1 es la que más importa para la calidad final de las respuestas, ya que determina si el chunk más relevante llega primero al LLM.

**Configuración elegida para generación:** Paragraph-aware (B) con `max_tokens=300`, `overlap_tokens=120`, embedder `BAAI/bge-m3`.

---

### 3. Resultados de generación

**Modelo LLM:** `google/gemma-2-2b-it` (2B parámetros, instruction-tuned), cargado en bfloat16 con `device_map="auto"`. Se usó `do_sample=False`, `max_new_tokens=400`, k=5 fragmentos de contexto.

| Modo | C (Correcta) | P (Parcial) | I (Incorrecta) | A (Alucinación) |
|---|---|---|---|---|
| **CON RAG** | **8 / 10 = 80%** | **2 / 10 = 20%** | 0 / 10 = 0% | 0 / 10 = 0% |
| **SIN RAG** | 2 / 10 = 20% | 0 / 10 = 0% | 0 / 10 = 0% | 8 / 10 = 80% |

**Cuadro detallado:**

| # | Pregunta | CON RAG | SIN RAG |
|---|---|---|---|
| P1 | ¿Qué es la AUH y a quiénes está destinada? | C | A |
| P2 | ¿Requisitos para tramitar la jubilación? | C | A |
| P3 | ¿A partir de qué semana se cobra la Asig. por Embarazo? | C | C |
| P4 | ¿Cómo se obtiene la Clave de la Seguridad Social? | C | A |
| P5 | ¿Qué es el CUIL y para qué se necesita? | C | C |
| P6 | ¿Qué documentos para la Asig. Familiar por Maternidad? | P | A |
| P7 | ¿Hasta qué edad cobran la AUH? | P | A |
| P8 | ¿Requisitos para la Prestación por Desempleo? | C | A |
| P9 | ¿Qué es la Pensión Universal para la Adulta Mayor? | C | A |
| P10 | ¿Qué trámites desde la app mi ANSES? | C | A |

---

#### Tres preguntas comparadas lado a lado

**P2 — ¿Cuáles son los requisitos para tramitar la jubilación en ANSES?**

🟢 **CON RAG:** *"Los requisitos para tramitar la jubilación en ANSES son: las mujeres deben tener 60 años, los hombres deben tener 65 años, ambos deben acreditar un mínimo de 30 años de aportes. En algunos casos, se requiere cumplir con otros requisitos según el régimen de jubilación."*

🔴 **SIN RAG:** *"Para tramitar la jubilación en ANSES es necesario: edad (65 años para jubilación por edad, 60 para anticipada con 30 años de cotización), estado de salud, documentos (DNI, certificado de nacimiento, certificado de cotización, certificado de discapacidad), proceso (en línea o presencial)."*

- Comentario: CON RAG responde con los datos exactos del corpus (60/65 años, 30 años de aportes). SIN RAG alucina requisitos que no existen en la guía ANSES: no hay requisito de "estado de salud" ni "certificado de cotización"; además confunde jubilación general con anticipada. Es el patrón típico de alucinación estructurada — respuesta que parece plausible pero es factualmente incorrecta.

---

**P9 — ¿Qué es la Pensión Universal para la Adulta Mayor?**

🟢 **CON RAG:** *"La Pensión Universal para la Adulta Mayor (PUAM) es una cobertura previsional para personas mayores de 65 años que no cuentan con ninguna jubilación o pensión. La prestación que brinda es del 80 por ciento del haber mínimo."*

🔴 **SIN RAG:** *"La Pensión Universal para la Adulta Mayor (PUAM) es un programa de asistencia financiera que busca garantizar un nivel mínimo de ingresos a los adultos mayores en España. Se otorga a personas mayores de 65 años que cumplan con ciertos requisitos económicos y sociales. El monto varía según la renta de la persona."*

- Comentario: CON RAG extrae correctamente los datos clave del corpus (65 años, 80% del haber mínimo). SIN RAG comete una alucinación geográfica: atribuye el programa a España, inventa que el monto varía según la renta, y omite el porcentaje exacto. Es el patrón más frecuente del modo SIN RAG en este corpus: el modelo conoce el concepto general pero lo asocia al sistema previsional de otro país.

---

**P6 — ¿Qué documentos se necesitan para cobrar la Asignación Familiar por Maternidad?**

🟢 **CON RAG:** *"No tengo información suficiente en el contexto proporcionado."*

🔴 **SIN RAG:** *"Para cobrar la Asignación Familiar por Maternidad necesitarás: certificado de nacimiento del bebé, documento de identidad del padre o madre, certificado de cobro de la Seguridad Social, declaración de ingresos, certificado de residencia, formulario de solicitud."*

- Comentario: Este es un caso donde CON RAG es más honesto que SIN RAG. La guía ANSES no incluye una lista de documentos para esta prestación — solo indica requisitos de elegibilidad (12 semanas de gestación, 3 meses de antigüedad laboral). CON RAG reconoce correctamente que el dato no está en el contexto. SIN RAG inventa una lista plausible pero no verificable, incluyendo documentos que la ANSES no solicita en este trámite.

---

### 4. Análisis de errores

**¿Cuándo falla el retrieval?**

El retrieval falló en un solo caso con Fixed-size (hit@1 = 0.90): una de las 10 preguntas no recuperó el documento correcto en primera posición. El problema ocurre cuando los chunks de ~500 caracteres mezclan texto de dos prestaciones distintas al cortar en el límite, haciendo que el embedding resultante represente un concepto híbrido con menor similitud a la query. Paragraph-aware evita esto al mantener cada bloque semántico íntegro.

**¿Cuándo alucina el LLM a pesar del RAG?**

En esta configuración final, el LLM no alucinó en ninguna de las 10 preguntas CON RAG. Los dos casos parciales (P6 y P7) no son alucinaciones sino limitaciones del corpus y del chunking:

- **P6 (documentos maternidad):** El PDF genuinamente no lista los documentos requeridos, solo los requisitos de elegibilidad. El modelo responde correctamente "No tengo información suficiente" — comportamiento esperado ante ausencia de datos en el contexto.
- **P7 (edad AUH):** El retrieval funcionó correctamente — el chunk con mayor score (0.729) es el correcto y llega primero. El problema es de chunking: ese chunk comienza con la Ayuda Escolar Anual ("a partir de los 45 días y hasta los 17 años inclusive") y luego arranca la descripción de la AUH, pero el límite de tokens lo corta antes de llegar al texto "menor de 18 años". El LLM lee el primer número relevante del chunk ("17 años") y lo usa como respuesta. El overlap de 120 tokens entre chunks no fue suficiente para incluir el dato clave de la prestación siguiente.

**Patrón SIN RAG:** El modelo sin RAG tiende a atribuir programas argentinos a España o a describir sistemas previsionales de otros países. En 8 de 10 preguntas generó información de otro sistema previsional. Esto sugiere que en su corpus de entrenamiento los datos sobre seguridad social española tienen mayor peso que los argentinos.

---

### 5. Conclusiones y trabajo futuro

**Qué funcionó bien:**

- El embedder `BAAI/bge-m3` con chunking paragraph-aware logró hit@1 = 1.00 en las 10 preguntas — retrieval sin errores a nivel de documento en este set de evaluación (8 documentos, 10 preguntas).
- El modelo `google/gemma-2-2b-it` siguió las instrucciones de grounding de forma consistente: 0 alucinaciones en 10 preguntas CON RAG.
- La comparación CON RAG vs SIN RAG es contundente: 80% correcto vs 20% correcto, y 0% alucinaciones vs 80% alucinaciones.


**Qué se haría diferente:**

- El set de evaluación de retrieval usa solo el nombre del documento como "doc esperado". Sería más preciso anotar el chunk específico correcto, y quizás así hacer un trazado más fino.
- La pregunta P6 (documentos de maternidad) es una pregunta que el corpus no puede responder — la guía ANSES omite esa información. En un sistema real habría que detectar estas preguntas y redirigir al usuario a la página oficial de ANSES.

**Qué se exploraría con más tiempo:**

- **LLM más grande:** probar `gemma-2-9b-it` para ver si los parciales (P6, P7) se resuelven con mayor capacidad de razonamiento.
- **Corpus expandido:** agregar las resoluciones oficiales de ANSES con los requisitos documentales completos, que en la guía actual están omitidos (ej: documentos para la Asignación Familiar por Maternidad).
- **Evaluación automática:** usar alguna especie de  LLM-as-judge para escalar la evaluación a más preguntas sin necesidad de anotación manual.
- **Mejor chunking:** Evaluar y mejorar el paragraph-aware que permitan traer la información completa a fin de responder correctamente. O probar alguna forma de chunking mejor para este problema.  
